
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Delta 테이블 만들기 및 작업 


## 환경 준비 (Databricks Free Edition)

- **카탈로그** `db_catalog`, **스키마** `db_schema` 를 사용합니다.
- 스키마 `db_schema` 아래 **`myfiles`** 볼륨을 만들고, **`employees.csv`** 를 업로드합니다.
- (다른 노트북에서 안내한 것과 동일한 데이터 레이아웃을 사용합니다.)



## A. 카탈로그 탐색

**Unity Catalog 개체 모델**

Unity Catalog에서는 모든 메타데이터가 메타스토어에 등록됩니다. Unity Catalog 메타스토어의 데이터베이스 개체 계층 구조는 테이블, 뷰, 볼륨, 모델 및 함수를 참조할 때 3단계 네임스페이스(catalog.schema.table-etc)로 표시되는 세 가지 수준으로 나뉩니다.

![](https://velog.velcdn.com/images/newnew_daddy/post/4e7dfbe4-a992-478d-8e2e-81da372f7362/image.png)


자세한 내용은 를 확인하세요.[Unity Catalog란 무엇입니까?](https://docs.databricks.com/en/data-governance/unity-catalog/index.html)

#### 1. 카탈로그 및 스키마 보기
강의 카탈로그 **db_catalog** 및 스키마를 수동으로 보려면 다음을 완료하십시오.

- 왼쪽 탐색 모음에서 카탈로그 아이콘을 선택합니다. 

- 카탈로그 이름(**db_catalog**)이 표시되어야 합니다.

- **db_catalog** 카탈로그를 펼치십시오. 카탈로그 내에 다양한 스키마(데이터베이스)가 표시되어야 합니다.

- 스키마 **db_schema** 를 찾습니다.

- 스키마를 확장합니다. 스키마에는 **myfiles**라는 볼륨만 포함되어 있습니다.

- 볼륨을 확장하십시오. 볼륨에는 이름이 지정된 **employees.csv** 단일 CSV 파일이 포함되어야 합니다.

#### 2. 카탈로그 및 스키마 기본값
셀을 실행하여 현재 카탈로그를 **db_catalog**로 설정하고 현재 스키마를 특정 스키마로 설정합니다. 이 구성을 사용하면 쿼리에서 2단계 명명 규칙(catalog.schema)을 사용할 필요가 없습니다. SELECT 문은 현재 카탈로그 및 스키마의 이름을 표시합니다. 

In [0]:
%sql
-- 현재 워크스페이스 기본 네임스페이스 확인
SELECT current_catalog(), current_schema();

`USE CATALOG` 및 `USE SCHEMA` 문을 사용하여 기본 카탈로그와 스키마를 바꿔 봅니다. 설정해 두면 테이블·뷰 등 객체를 지정할 때 매번 **catalog.schema.object** 형태의 3단 이름을 쓰지 않아도 됩니다.

  - `USE CATALOG` – 현재 카탈로그를 설정합니다.

  - `USE SCHEMA` – 현재 스키마를 설정합니다.

    **참고:** 동적 식별자가 필요할 때는 `IDENTIFIER('이름')` 과 같이 [IDENTIFIER 절](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-names-identifier-clause?language=SQL)을 쓸 수 있습니다. 이 실습에서는 스키마 이름을 **db_schema** 로 고정해 사용합니다.

다음 셀을 실행하여 기본 카탈로그를 **db_catalog**, 스키마를 **db_schema** 로 맞춘 뒤 `current_catalog()`, `current_schema()` 로 확인합니다.

In [0]:
%sql
USE CATALOG db_catalog;
USE SCHEMA db_schema;

SELECT current_catalog(), current_schema()

#### 3. 스키마 설명 
[DESCRIBE SCHEMA EXTENDED](https://docs.databricks.com/en/sql/language-manual/sql-ref-syntax-aux-describe-schema.html) 문을 사용하여 스키마의 메타데이터 및 속성을 표시합니다.

In [0]:
%sql
DESCRIBE SCHEMA EXTENDED db_schema

#### 4. 테이블 표시
[SHOW TABLES](https://docs.databricks.com/en/sql/language-manual/sql-ref-syntax-aux-show-tables.html) 문을 사용하여 스키마에서 사용 가능한 테이블을 표시합니다. 현재 스키마에 사용할 수 있는 테이블이 없음을 알 수 있습니다.

In [0]:
%sql
SHOW TABLES;

#### 5. 볼륨 표시
[SHOW VOLUMES](https://docs.databricks.com/en/sql/language-manual/sql-ref-syntax-aux-show-volumes.html) 문을 사용하여 스키마에서 사용할 수 있는 모든 볼륨을 나열합니다. 사용 가능한 단일 볼륨인 **myfiles**가 있습니다.


**참고**: 볼륨은 클라우드 오브젝트 스토리지 위치의 스토리지 논리 볼륨을 나타내는 Unity Catalog 오브젝트입니다. 볼륨은 파일에 액세스, 저장, 관리 및 구성하는 기능을 제공합니다. 테이블은 테이블 형식 데이터 세트에 대한 거버넌스를 제공하는 반면, 볼륨은 테이블 형식이 아닌 데이터 세트에 대한 거버넌스를 추가합니다. 

볼륨을 사용하여 정형, 반정형 및 비정형 데이터를 포함한 모든 형식의 파일을 저장하고 액세스할 수 있습니다.

[Unity Catalog 볼륨이란 무엇입니까?](https://docs.databricks.com/en/volumes/index.html)

In [0]:
%sql
SHOW VOLUMES;

#### 6. 볼륨의 파일/데이터 나열
LIST문을 사용하여 **myfiles** 볼륨의 모든 파일을 나열하십시오. 볼륨에는 **employees.csv** 파일만 있습니다. 여기에는 현재 직원에 대한 정보가 포함되어 있습니다. 

볼륨의 데이터와 상호 작용할 때 항상 다음 형식을 갖는 Unity Catalog에서 제공하는 경로를 사용합니다. `/Volumes/catalog_name/schema_name/volume_name/`.

Unity Catalog 볼륨으로 관리되는 디렉터리 및 데이터 파일 탐색에 대한 자세한 내용은 [스토리지 탐색 및 데이터 파일 찾기](https://docs.databricks.com/en/discover/files.html)  설명서.


In [0]:
spark.sql(f"LIST '/Volumes/db_catalog/db_schema/myfiles'").display()

## B. CSV 파일에서 Delta 테이블 만들기

Databricks에 저장된 모든 테이블은 기본적으로 Delta 테이블입니다. Apache Spark DataFrames를 사용하든 SQL를 사용하든 기본 설정을 사용하여 레이크하우스에 데이터를 저장하기만 하면 Delta Lake의 모든 이점을 얻을 수 있습니다.

1. SQL를 사용하여 **employees.csv** 파일을 직접 쿼리하여 파일을 봅니다. 쿼리는 헤더가 데이터의 첫 번째 행으로 있는 CSV 파일의 뷰를 반환합니다.

**참고**: 이 구문은 Spark SQL에만 해당되며 먼저 테이블에 명시적으로 로드하지 않고도 파일을 직접 읽을 수 있습니다. 파일 형식을 지정하고 파일 경로를 **백틱**으로 묶습니다. 이 방법은 다양한 파일 형식에 적용됩니다.

*SELECT * FROM \<file_format>.\`/path/to/file`;*

In [0]:
spark.sql(f'''
SELECT *
FROM csv.`/Volumes/db_catalog/db_schema/myfiles/` 
''').display()

2. *text* 파일 형식을 사용하여 파일을 열의 문자열로 볼 수 있습니다. 이렇게 하면 파일의 내용을 볼 수 있습니다. 첫 번째 행에는 열 이름이 포함되어 있고 필드는 쉼표로 구분됩니다.

In [0]:
spark.sql(f'''
SELECT * 
FROM text.`/Volumes/db_catalog/db_schema/myfiles/`
''').display()

3. SQL를 사용하면 [read_files](https://docs.databricks.com/en/sql/language-manual/functions/read_files.html) 테이블 반환 함수를 사용하여 CSV 파일을 표 형식으로 읽고 특정 옵션을 적용하여 파일을 읽는 방법을 수정할 수 있습니다. 

    쿼리를 실행하고 결과에 유효한 열 이름을 가진 4명의 직원이 표시되는지 확인합니다.

**참고**: 복구된 데이터 열은 스키마와 일치하지 않는 모든 데이터를 복구하기 위해 기본적으로 제공됩니다.

In [0]:
%sql
SELECT *
FROM read_files(
  '/Volumes/db_catalog/db_schema/myfiles/',
  format => 'csv',
  header => true,
  inferSchema => true
  )

4. 위의 쿼리와 함께 CREATE TABLE AS(CTAS) 문을 사용하여 파일을 사용하여 **current_employees** 명명된 **employees.csv** 레이크하우스에 Delta 테이블을 만듭니다. 테이블이 스키마에 생성됩니다.

**참고:** CREATE TABLE 문은 기본적으로 Delta 테이블을 만듭니다.

In [0]:
%sql
-- 시연 목적으로 그것이 이미 있는 경우 테이블을 삭제합니다.
DROP TABLE IF EXISTS current_employees;

-- CSV 파일을 사용하여 Delta 테이블 만들기

CREATE TABLE current_employees AS
SELECT 
  ID, 
  FirstName, 
  Country, 
  Role 
FROM read_files(
  '/Volumes/db_catalog/db_schema/myfiles/',
  format => 'csv',
  header => true,
  inferSchema => true
  );


-- 테이블 표시
SELECT *
FROM current_employees;

5. 스키마에서 테이블을 수동으로 보려면 다음 단계를 완료하십시오.

   a. 왼쪽 탐색 모음에서 카탈로그 아이콘을 ![catalog_icon](../Includes/images/catalog_icon.png) 선택합니다. 

   b. 카탈로그 이름 **db_catalog**를 찾습니다.

   c. 새로 고침 아이콘을 ![refresh_icon](../Includes/images/refresh_icon.png) 선택하여 카탈로그를 새로 고칩니다.

   d. **db_catalog** 카탈로그를 확장합니다. 카탈로그 내에서 다양한 스키마(데이터베이스)가 표시되어야 합니다.

   e. 스키마 **db_schema** 를 찾습니다. 

   f. 스키마를 확장합니다. 스키마에 **볼륨**과 **테이블**이 포함되어 있습니다.

   g. **테이블**을 확장합니다. **current_employees** Delta 테이블이 사용 가능한지 확인합니다.

(선택 사항) Python을 사용하여 CSV 파일에서 Delta 테이블을 만들려면 아래 코드를 사용할 수 있습니다.

In [0]:
#
# CSV 파일을 읽고 Spark DataFrame을 생성합니다.
#
sdf = (spark
       .read
       .format("csv")
       .option('header', 'true')
       .option('inferSchema','true')
       .load(f'/Volumes/db_catalog/db_schema/myfiles/')
    )


#
# Spark DataFrame에서 Delta 테이블 만들기
#

(sdf
 .write
 .mode("overwrite")
 .format("delta")
 .saveAsTable(f"db_catalog.db_schema.current_employees_py")
)

Python을 사용하여 Delta 테이블을 읽습니다.

In [0]:
(spark
 .read
 .table(f"db_catalog.db_schema.current_employees_py")
 .display()
)

6. Python를 사용하여 스키마의 테이블을 표시합니다. **current_employees** 및 **current_employees_py**라는 테이블이 표시되어야 합니다.


In [0]:
spark.catalog.listTables("db_schema")

7. 테이블을 **current_employees** 쿼리하여 데이터를 봅니다. 그것에 현재 직원 목록과 함께 4행의 데이터가 포함되어 있는지 확인합니다.

In [0]:
%sql
SELECT * 
FROM current_employees;

8. DESCRIBE DETAIL문을 사용하여 **current_employees** 테이블에 대한 자세한 정보를 볼 수 있습니다. 결과를 봅니다. 

    다음 사항에 유의하십시오.
    - **형식** 열은 **current_employees** 테이블이 Delta 테이블로 작성되었음을 표시합니다.
    - **위치** 열은 AWS에 대해 다음 형식으로 테이블의 클라우드 위치를 표시합니다. 
        - *s3://\<bucket-name>/\<metastore id>/tables/\<table id>* 

**참고:** 
    - DESCRIBE DETAIL문의 결과에는 Delta 테이블에 대한 추가 정보가 포함됩니다. 자세한 내용은 [상세 스키마 문서](https://docs.databricks.com/en/delta/table-details.html#detail-schema)를 참고하세요.
    - Unity Catalog에서 관리하는 데이터의 경로 작동 방식에 대한 자세한 내용은 [Unity Catalog에서 관리하는 데이터의 경로는 어떻게 작동하나요?](https://docs.databricks.com/en/data-governance/unity-catalog/paths.html) 문서.

In [0]:
%sql
DESCRIBE DETAIL current_employees;

9. [DESCRIBE EXTENDED](https://docs.databricks.com/en/sql/language-manual/sql-ref-syntax-aux-describe-table.html) 문을 사용하여 Delta 테이블에 대한 지정된 열 및 추가 메타데이터에 대한 자세한 정보를 표시합니다.

    다음 사항에 유의하십시오.
    - 결과의 맨 위 부분에는 테이블의 열 메타데이터가 표시됩니다.
    - 열을 **col_name** 아래로 스크롤하여 *유형* 값을 찾습니다. **data_type** 열이 *Managed*를 지정하는 것을 확인합니다. 이는 Databricks가 테이블의 수명 주기 및 파일 레이아웃을 관리한다는 것을 나타냅니다. 관리되는 테이블은 기본 테이블 생성 방법입니다.



In [0]:
%sql
DESCRIBE EXTENDED current_employees;

10. **current_employees** 테이블에 대해 [DESCRIBE HISTORY](https://docs.databricks.com/en/sql/language-manual/delta-describe-history.html) 문을 실행하여 델타 테이블의 이력을 조회합니다.

    다음 사항에 유의하십시오.
    - **version** 열은 테이블이 버전 0에 있음을 표시합니다.
    - **timestamp** 열은 테이블이 생성된 시점을 표시합니다.
    - **operation** 열은 수행된 작업을 표시합니다.
    - **operationsMetrics** 열은 파일 수, 출력 행 수 및 출력 바이트 수에 대한 정보를 표시합니다.

In [0]:
%sql
DESCRIBE HISTORY current_employees;

## C. Delta 테이블의 레코드 삽입, 업데이트 및 삭제

1. 현재 **current_employees** 테이블을 봅니다. 

    Delta 표에서 다음 사항에 유의하십시오.
    - 4개의 열과 4개의 행이 있습니다.
    - **ID** 값 *1111*, *2222*, *3333*, *4444*는 테이블에 있습니다.
    - **ID** *1111*의 **역할**은 *관리자입니다*.

In [0]:
%sql
SELECT * 
FROM current_employees;

2. **current_employees** Delta 테이블에서 다음 작업을 수행합니다.

   a. Alex*와 *Sanjay*라는 두 명의 신입 사원을 삽입합니다.* 

   b. 직원 **ID** *1111*의 **역할**을 *선임 관리자*로 업데이트합니다.

   c. **ID** 3333인 직원의 기록을 삭제합니다.

In [0]:
%sql
-- 1. 테이블에 두 명의 직원을 삽입합니다.
INSERT INTO current_employees 
VALUES
    (5555, 'Alex','미국', '강사'),
    (6666, 'Sanjay','인도', '강사');

-- 2. 테이블의 레코드 업데이트
UPDATE current_employees
  SET Role = '선임 관리자'
  WHERE ID = 1111;

-- 3. 테이블에서 레코드 삭제
DELETE FROM current_employees
  WHERE ID = 3333;

3. 테이블의 **current_employees** 데이터를 봅니다. 테이블이 원래 버전에서 수정되었습니다.

In [0]:
%sql
SELECT *
FROM current_employees
ORDER BY ID;

4. Delta Lake 테이블을 수정하는 각 작업은 새 테이블 버전을 만듭니다. 테이블의 이력을 봅니다.

    다음 사항에 유의하십시오.
    - 테이블에는 버전 0 - 4가 있습니다. 
    - **버전 0** 은 생성된 원래 테이블입니다.
    - **버전 1** 에는 두 명의 새 직원을 삽입한 WRITE 작업이 포함되어 있습니다.
    - **버전 2** 에는 작업 역할을 수정한 UPDATE 작업이 포함되어 있습니다.
    - **버전 3** 에는 직원을 제거한 DELETE 작업이 포함되어 있습니다.
    - **버전 4** 에는 테이블에 대한 OPTIMIZE 작업이 포함되어 있습니다. 예측 최적화는 Databricks의 Delta Lake에 있는 기능으로, Delta 테이블을 자동으로 최적화합니다(선택적 기능). 자세한 정보를 원하시면 [Delta Lake에 대한 예측 최적화를 참조하세요.
    ](https://docs.databricks.com/en/optimizations/predictive-optimization.html) 문서.

**참고:**  `OPTIMIZE` 작업은 순서가 다를 수 있습니다.

In [0]:
%sql
DESCRIBE HISTORY current_employees;

## D. 시간 이동을 사용하여 이전 버전의 Delta 테이블 읽습니다.
이력 정보를 사용하여 운영을 감사하거나, 테이블을 롤백하거나, 시간 이동을 사용하여 특정 시점에 테이블을 쿼리할 수 있습니다.

1. 테이블의 **current_employees** 현재 버전을 봅니다. 기본적으로 테이블의 최신 버전이 사용됩니다.

In [0]:
%sql
SELECT *
FROM current_employees
ORDER BY ID;

2. 시간 이동을 사용하여 DELETE 작업 전에 테이블을 뷰합니다. 결과에는 레코드가 삭제되기 전의 직원 6명이 표시됩니다.

**참고**: 시간 이동은 Delta Lake 트랜잭션 로그의 기능을 활용하여 더 이상 테이블에 없는 데이터에 액세스합니다.

In [0]:
%sql
SELECT *
FROM current_employees VERSION AS OF 2
ORDER BY ID;

-- 대체 구문
-- SELECT *
-- FROM current_employees@v2
-- ORDER BY ID;

3. 버전 0을 사용하여 원본 테이블을 봅니다. 원래 4명의 직원이 표시됩니다.

In [0]:
%sql
SELECT *
FROM current_employees VERSION AS OF 0
ORDER BY ID;

## E. 테이블 삭제
1. 이 데모에서 만든 Delta 테이블을 삭제합니다.

In [0]:
%sql
DROP TABLE IF EXISTS current_employees;
DROP TABLE IF EXISTS current_employees_py;

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>